# 11 · Export results for the app
**Question:** what small, safe files does the app need so it opens instantly with the real X5 results?

**What this notebook does**
- **Part A, recomputed:** rebuilds the Step 8 locked exam (40,008 X5 customers never used for building) and saves the *ingredients* of the profit curves: for every 1% of customers contacted (best first), how many contacts and how many extra buyers, for 3 ways of ranking (uplift, likely-buyer, random). The app turns these into rupees live, so every slider (margin, channel cost, budget) works instantly. The bootstrap ranges use the same random draws as Step 8, so the ranges match too.
- **Part B, read back:** heavy results (model leaderboard, personas, offers, drift, exploration, contrast) are read from the tables already saved inside notebooks 04-10. Nothing is re-trained, so the numbers are exactly the ones in those notebooks.
- **Part C:** small chart data for the "How it was built" page, and the assumptions with their sources.
- **Last cell:** checks that the headline numbers are reproduced and that no file holds customer rows.

**Honesty rule:** only aggregates leave this notebook. No client ids, no customer rows from X5, Starbucks or Complete Journey.

## Cell 0 · Setup
Paths, the Step 8 money assumptions (copied, not changed), and two helpers: one reads a table saved inside another notebook, one saves a file and reports its size.

In [1]:
import pandas as pd, numpy as np, duckdb, json, re, time, warnings
warnings.filterwarnings('ignore')
import lightgbm as lgb
from io import StringIO
from pathlib import Path
from nudge_engine import check_balance, budget_plan, profit_curve

X5 = Path('..') / 'data_x5' / 'features'
OUT = Path('..') / 'app' / 'results'; (OUT / 'tables').mkdir(parents=True, exist_ok=True)

# Step 8 assumptions (unchanged)
BASKET, GROSS_MARGIN = 1000, 0.15
MARGIN = BASKET * GROSS_MARGIN                     # ₹150 profit per extra purchase
CHANNELS = {'Email': 0.01, 'SMS': 0.10, 'WhatsApp': 1.02, 'AI voice call': 3.33}
PER = 100_000                                      # everything per 1 lakh customers

def saved_tables(notebook, cell):
    # Tables displayed in a saved notebook cell, with pandas' two-row headers flattened.
    c = json.load(open(notebook, encoding='utf-8'))['cells'][cell]
    out = []
    for o in c.get('outputs', []):
        if 'text/html' not in o.get('data', {}): continue
        t = pd.read_html(StringIO(''.join(o['data']['text/html'])))[0]
        if isinstance(t.columns, pd.MultiIndex):
            t.columns = [next((p for p in col if not str(p).startswith('Unnamed')), '') for col in t.columns]
        if str(t.columns[0]).startswith('Unnamed') or t.columns[0] == '':
            first = t.iloc[:, 0]
            if first.tolist() == list(range(len(t))):
                t = t.drop(columns=t.columns[0])     # plain 0,1,2 row numbers
            else:
                t = t.rename(columns={t.columns[0]: 'row'})   # row labels without a name (e.g. measure names)
        out.append(t)
    return out

def printed(notebook, cell):
    c = json.load(open(notebook, encoding='utf-8'))['cells'][cell]
    return ''.join(''.join(o.get('text', '')) for o in c.get('outputs', []) if o.get('name') == 'stdout')

SAVED = []
def save(obj, name):
    p = OUT / name
    if isinstance(obj, pd.DataFrame): obj.to_csv(p, index=False)
    elif name.endswith('.npz'): np.savez_compressed(p, **obj)
    else: p.write_text(json.dumps(obj, indent=1, ensure_ascii=False, default=float), encoding='utf-8')
    SAVED.append(p); print(f'saved {p.relative_to(OUT.parent)} ({p.stat().st_size / 1024:,.0f} KB)')

## Part A · Cell 1 · Rebuild the Step 8 locked exam
Exactly as Step 8: the 80% build customers train the **likely-buyer model** (who buys when messaged, the usual approach); the locked 20% exam is scored by the uplift champion, the likely-buyer model and a random order.

In [2]:
t0 = time.time()
prof = duckdb.sql(f"SELECT * FROM '{(X5 / 'customer_profiles.parquet').as_posix()}' WHERE in_experiment = 1").df()
prof = pd.concat([prof, pd.get_dummies(prof.gender, prefix='gender').astype(int)], axis=1)
XC = [c for c in prof.columns if c not in {'client_id', 'gender', 'treatment_flg', 'target', 'in_experiment'}]
prof[XC] = prof[XC].astype('float32')
hs = duckdb.sql(f"SELECT client_id, champion, step3_tlearner FROM '{(X5 / 'holdout_scores.parquet').as_posix()}'").df()
hold = prof.merge(hs, on='client_id'); dev = prof[~prof.client_id.isin(hs.client_id)]
dt = dev[dev.treatment_flg == 1]
likely = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.03, num_leaves=15, min_child_samples=400, subsample=0.8, subsample_freq=1,
                            colsample_bytree=0.7, reg_lambda=5.0, verbose=-1, random_state=42).fit(dt[XC], dt.target)
SCORES = {'uplift': hold.champion.values, 'likely': likely.predict_proba(hold[XC])[:, 1],
          'random': np.random.default_rng(0).random(len(hold))}
T, Y, SCALE = hold.treatment_flg.values, hold.target.values.astype(float), PER / len(hold)
print(f'locked exam {len(hold):,} customers (scaled x{SCALE:.3f} to 1 lakh) | built in {time.time() - t0:.0f}s')

locked exam 40,008 customers (scaled x2.500 to 1 lakh) | built in 4s


## Part A · Cell 2 · Profit-curve ingredients
For each share contacted (0%, 1%, ... 100%, best first): **contacts** and **extra buyers** (buy rate if contacted minus buy rate if held back, times the group size), per 1 lakh customers.
Profit is then simply `extra buyers x profit per purchase - contacts x cost per contact`, which the app computes live.

Two kinds of uncertainty are saved:
- **single-curve ranges:** the same 200 bootstrap draws as Step 8 (`profit_curve`, seed 1), so ranges match the notebook;
- **paired ranges:** 300 draws that resample the *same customers* for all three rankings (as Step 8 did for ₹20k), so the app can show a range for "uplift minus likely-buyer" at any budget.

In [3]:
SHARES = np.round(np.arange(0, 1.0001, 0.01), 2); n = len(hold); KS = np.round(SHARES * n).astype(int)

def ingredients(score, w):
    # extra buyers and contacts at each share, customers weighted by w (in score order)
    o = np.argsort(-score, kind='stable'); t, y = T[o], Y[o]
    ct, cc = np.cumsum(w * t), np.cumsum(w * (1 - t)); yt, yc = np.cumsum(w * y * t), np.cumsum(w * y * (1 - t)); sz = np.cumsum(w)
    extra, cont = np.zeros(len(KS)), np.zeros(len(KS))
    for j, k in enumerate(KS):
        if k == 0: continue
        i = k - 1; lift = yt[i] / max(ct[i], 1e-9) - yc[i] / max(cc[i], 1e-9)
        extra[j], cont[j] = lift * sz[i] * SCALE, sz[i] * SCALE
    return extra, cont

t0 = time.time(); rows, boots = [], {}
for m, s in SCORES.items():
    e, c = ingredients(s, np.ones(n)); rows.append(pd.DataFrame({'method': m, 'share': SHARES, 'contacts': c, 'extra_buyers': e}))
    rng = np.random.default_rng(1)                       # same draws as profit_curve(..., n_boot=200, seed=1)
    B = [ingredients(s, np.bincount(rng.integers(0, n, n), minlength=n).astype(float)) for _ in range(200)]
    boots[f'{m}_extra'] = np.array([b[0] for b in B], dtype='float32'); boots[f'{m}_contacts'] = np.array([b[1] for b in B], dtype='float32')
rng = np.random.default_rng(7); P = {m: [] for m in SCORES}
for _ in range(300):                                     # paired: same resampled customers for every ranking
    i = rng.integers(0, n, n)
    for m, s in SCORES.items():
        o = i[np.argsort(-s[i], kind='stable')]; t, y = T[o], Y[o]
        ct, yt, yc = np.cumsum(t), np.cumsum(y * t), np.cumsum(y * (1 - t)); cc = np.arange(1, n + 1) - ct
        P[m].append([0.0 if k == 0 else (yt[k - 1] / max(ct[k - 1], 1) - yc[k - 1] / max(cc[k - 1], 1)) * k * SCALE for k in KS])
for m in SCORES: boots[f'{m}_paired_extra'] = np.array(P[m], dtype='float32')
CURVES = pd.concat(rows, ignore_index=True)
print(f'ingredients ready in {time.time() - t0:.0f}s')
save(CURVES.round(3), 'x5_curves.csv'); save(boots, 'x5_boots.npz')

ingredients ready in 10s
saved results\x5_curves.csv (9 KB)
saved results\x5_boots.npz (717 KB)


## Part A · Cell 3 · Check: do the ingredients give back the Step 8 rupees?
Re-create three Step 8 results from the saved ingredients alone. If these match, the app's money pages will match the notebooks.

In [4]:
def profit(m, margin, cost):
    c = CURVES[CURVES.method == m].reset_index(drop=True)
    return c.assign(profit=c.extra_buyers * margin - c.contacts * cost)

REPRO = {}
sms = profit('uplift', MARGIN, CHANNELS['SMS']); s5 = sms[np.isclose(sms.share * 20, np.round(sms.share * 20))]   # Step 8 used 5% steps here
pk = s5.loc[s5.profit.idxmax()]
pb = boots['uplift_extra'] * MARGIN - boots['uplift_contacts'] * CHANNELS['SMS']
lo, hi = np.percentile(pb[:, int(pk.name)], [2.5, 97.5])
REPRO['SMS best share'] = (f'{pk.share:.0%}', '95%'); REPRO['SMS best profit ₹'] = (round(pk.profit), 495972)
REPRO['SMS range low ₹'] = (round(lo), 332402); REPRO['SMS range high ₹'] = (round(hi), 623614)
wa = CHANNELS['WhatsApp']
for m, ref in [('uplift', 243600), ('likely', 3300)]:
    REPRO[f'₹20k WhatsApp, {m} ₹'] = (round(budget_plan(profit(m, MARGIN, wa), 20_000, wa).profit, -2), ref)
for k, (got, want) in REPRO.items(): print(f"{'PASS' if got == want else 'CHECK'}  {k}: {got} (Step 8: {want})")

PASS  SMS best share: 95% (Step 8: 95%)
PASS  SMS best profit ₹: 495972 (Step 8: 495972)
PASS  SMS range low ₹: 332402 (Step 8: 332402)
PASS  SMS range high ₹: 623614 (Step 8: 623614)
PASS  ₹20k WhatsApp, uplift ₹: 243600.0 (Step 8: 243600)
PASS  ₹20k WhatsApp, likely ₹: 3300.0 (Step 8: 3300)


## Part A · Cell 4 · Headline numbers from the random test
Average effect of the message, the fairness check, naive vs fair report, the decision flip, the lift ladder, and "top 10% respond 3.5x the average". The app's Prove page recomputes the reports from the buy rates saved here.

In [5]:
ex_t, ex_c = prof[prof.treatment_flg == 1].target, prof[prof.treatment_flg == 0].target
LIFT = ex_t.mean() - ex_c.mean(); se = np.sqrt(ex_t.var() / len(ex_t) + ex_c.var() / len(ex_c))
bal = check_balance(prof, [c for c in XC if not c.startswith('gender_')] + ['gender_F', 'gender_M', 'gender_U'], 'treatment_flg', 0)[1].abs()

def rates(sel):
    g = hold[sel]; t, c = g[g.treatment_flg == 1].target, g[g.treatment_flg == 0].target
    return {'share': float(sel.mean()), 'rate_contacted': t.mean(), 'rate_held_back': c.mean(),
            'n_contacted': len(t), 'n_held_back': len(c), 'se_lift': np.sqrt(t.var() / len(t) + c.var() / len(c))}

pct = hold.champion.rank(pct=True)
third = pd.qcut(hold.champion.rank(method='first'), 3, labels=['bottom', 'middle', 'top'])
lift_hold = Y[T == 1].mean() - Y[T == 0].mean(); top10 = rates(hold.champion >= hold.champion.quantile(0.9))
SUMMARY = {
    'experiment_customers': len(prof), 'all_customers': 400_162, 'purchases_millions': 45.8, 'exam_customers': len(hold), 'measures': 66,
    'buy_rate_held_back': ex_c.mean(), 'buy_rate_messaged': ex_t.mean(), 'lift': LIFT, 'lift_low95': LIFT - 1.96 * se, 'lift_high95': LIFT + 1.96 * se,
    'largest_balance_gap': float(bal.max()), 'balance_top10': bal.sort_values(ascending=False).head(10).round(4).to_dict(),
    'exam_lift': lift_hold,
    'top30_by_uplift': rates(pct > 0.7), 'top_half': rates(pct > 0.5), 'bottom_half': rates(pct <= 0.5),
    'thirds': {k: rates((third == k).values) for k in ['bottom', 'middle', 'top']},
    'top10': top10, 'top10_vs_average': (top10['rate_contacted'] - top10['rate_held_back']) / lift_hold,
}
t30 = SUMMARY['top30_by_uplift']; naive, fair = t30['rate_contacted'] * 30_000, (t30['rate_contacted'] - t30['rate_held_back']) * 30_000
bh = SUMMARY['bottom_half']; flip = (bh['rate_contacted'] - bh['rate_held_back']) * MARGIN - CHANNELS['AI voice call']
print(f"message lift {100 * LIFT:+.1f} pts (95% {100 * SUMMARY['lift_low95']:+.1f} to {100 * SUMMARY['lift_high95']:+.1f}) | largest balance gap {SUMMARY['largest_balance_gap']:.4f}")
print(f'naive {naive:,.0f} vs fair {fair:,.0f} buyers -> overstated {naive / fair:.1f}x | bottom-half AI call: naive ₹{bh["rate_contacted"] * MARGIN - 3.33:.0f}, fair ₹{flip:.2f}')
print(f"top 10% respond {SUMMARY['top10_vs_average']:.1f}x the average | ladder (pts): " + ' / '.join(f"{100 * (v['rate_contacted'] - v['rate_held_back']):.2f}" for v in SUMMARY['thirds'].values()))
save(SUMMARY, 'x5_summary.json')

message lift +3.3 pts (95% +2.9 to +3.7) | largest balance gap 0.0125
naive 15,650 vs fair 1,818 buyers -> overstated 8.6x | bottom-half AI call: naive ₹105, fair ₹-0.47
top 10% respond 3.5x the average | ladder (pts): 0.95 / 2.88 / 5.65
saved results\x5_summary.json (2 KB)


## Part B · Cell 5 · Read back the saved tables from notebooks 04-10
Each entry says which notebook and cell the table comes from, so every number in the app can be traced back. These are all aggregates (averages, rates, counts by group).

In [6]:
SOURCES = {   # name: (notebook, cell, which table in that cell)
    'step4_leaderboard': ('04_model_selection.ipynb', 7, 0), 'step4_exam': ('04_model_selection.ipynb', 10, 0),
    'step4_share_of_extra_buyers': ('04_model_selection.ipynb', 10, 1),
    'step5_top20': ('05_who_to_message.ipynb', 4, 0), 'step5_by_share': ('05_who_to_message.ipynb', 8, 0),
    'step5_segments': ('05_who_to_message.ipynb', 14, 0), 'step5_personas': ('05_who_to_message.ipynb', 14, 1),
    'step6_starbucks_offer_types': ('06_offer_types.ipynb', 10, 0), 'step6_starbucks_offers': ('06_offer_types.ipynb', 12, 0),
    'step6_starbucks_best_by_group': ('06_offer_types.ipynb', 14, 0), 'step6_hillstrom_effects': ('06_offer_types.ipynb', 18, 0),
    'step6_hillstrom_by_group': ('06_offer_types.ipynb', 20, 0),
    'step7_x5_by_cost': ('07_engine.ipynb', 5, 0), 'step7_summary': ('07_engine.ipynb', 13, 0),
    'step8_channel_ladder': ('08_money.ipynb', 6, 0), 'step8_whatsapp_budget': ('08_money.ipynb', 8, 0),
    'step8_offer_economics': ('08_money.ipynb', 10, 0), 'step8_offer_effects': ('08_money.ipynb', 10, 1),
    'step8_starbucks_policies': ('08_money.ipynb', 12, 0), 'step8_starbucks_budget': ('08_money.ipynb', 14, 0),
    'step8_breakeven': ('08_money.ipynb', 16, 0), 'step8_stress_test': ('08_money.ipynb', 18, 0),
    'step9_naive_fair': ('09_measure.ipynb', 4, 0), 'step9_decision_flip': ('09_measure.ipynb', 6, 0),
    'step9_sample_size': ('09_measure.ipynb', 8, 0), 'step9_control_group': ('09_measure.ipynb', 10, 0),
    'step9_weeks_needed': ('09_measure.ipynb', 12, 0), 'step9_exploration': ('09_measure.ipynb', 14, 0),
    'step9_recorded_chance': ('09_measure.ipynb', 18, 0), 'step9_drift': ('09_measure.ipynb', 21, 0),
    'step9_monthly_behaviour': ('09_measure.ipynb', 21, 1), 'step9_lift_ladder': ('09_measure.ipynb', 23, 0),
    'step9_send_time': ('09_measure.ipynb', 25, 0), 'step9_playbook': ('09_measure.ipynb', 27, 0),
    'step10_naive': ('10_contrast.ipynb', 4, 0), 'step10_careful': ('10_contrast.ipynb', 6, 0), 'step10_summary': ('10_contrast.ipynb', 8, 0),
}
TABLES = {}
for name, (nb, cell, k) in SOURCES.items():
    TABLES[name] = saved_tables(nb, cell)[k]; save(TABLES[name], f'tables/{name}.csv')

# a few results exist only as printed sentences: keep them word for word
LINES = {'step5_uplift_vs_likely_20pct': printed('05_who_to_message.ipynb', 6), 'step5_rules_vs_champion': printed('05_who_to_message.ipynb', 14),
         'step8_20k_difference': printed('08_money.ipynb', 8), 'step8_sms_peak': printed('08_money.ipynb', 4),
         'step9_formula_vs_simulation': printed('09_measure.ipynb', 8), 'step9_targeted_only_cost': printed('09_measure.ipynb', 16),
         'step9_drift_alarm_calibration': printed('09_measure.ipynb', 20), 'step4_improvement': printed('04_model_selection.ipynb', 10)}
LINES = {k: v.strip() for k, v in LINES.items()}
save(LINES, 'printed_results.json')

saved results\tables\step4_leaderboard.csv (1 KB)
saved results\tables\step4_exam.csv (0 KB)
saved results\tables\step4_share_of_extra_buyers.csv (0 KB)
saved results\tables\step5_top20.csv (0 KB)
saved results\tables\step5_by_share.csv (0 KB)
saved results\tables\step5_segments.csv (0 KB)


saved results\tables\step5_personas.csv (0 KB)
saved results\tables\step6_starbucks_offer_types.csv (0 KB)
saved results\tables\step6_starbucks_offers.csv (1 KB)
saved results\tables\step6_starbucks_best_by_group.csv (1 KB)
saved results\tables\step6_hillstrom_effects.csv (0 KB)
saved results\tables\step6_hillstrom_by_group.csv (1 KB)


saved results\tables\step7_x5_by_cost.csv (1 KB)
saved results\tables\step7_summary.csv (0 KB)
saved results\tables\step8_channel_ladder.csv (0 KB)
saved results\tables\step8_whatsapp_budget.csv (0 KB)
saved results\tables\step8_offer_economics.csv (0 KB)


saved results\tables\step8_offer_effects.csv (0 KB)
saved results\tables\step8_starbucks_policies.csv (1 KB)
saved results\tables\step8_starbucks_budget.csv (0 KB)
saved results\tables\step8_breakeven.csv (0 KB)
saved results\tables\step8_stress_test.csv (0 KB)


saved results\tables\step9_naive_fair.csv (0 KB)
saved results\tables\step9_decision_flip.csv (0 KB)
saved results\tables\step9_sample_size.csv (0 KB)
saved results\tables\step9_control_group.csv (0 KB)
saved results\tables\step9_weeks_needed.csv (0 KB)
saved results\tables\step9_exploration.csv (1 KB)
saved results\tables\step9_recorded_chance.csv (0 KB)


saved results\tables\step9_drift.csv (0 KB)
saved results\tables\step9_monthly_behaviour.csv (0 KB)
saved results\tables\step9_lift_ladder.csv (0 KB)
saved results\tables\step9_send_time.csv (0 KB)
saved results\tables\step9_playbook.csv (1 KB)
saved results\tables\step10_naive.csv (0 KB)
saved results\tables\step10_careful.csv (0 KB)


saved results\tables\step10_summary.csv (0 KB)
saved results\printed_results.json (1 KB)


## Part C · Cell 6 · Chart data for "How it was built"
Small aggregate series: the Qini curves of the first model (Step 3) and the champion (Step 4) on the locked exam, and the Complete Journey weekly spend of three groups (Step 10).

In [7]:
def qini_points(score, k=50):
    # cumulative extra buyers (per customer in the exam) when contacting the top x% by score
    o = np.argsort(-score, kind='stable'); t, y = T[o], Y[o]
    nt, nc, yt, yc = np.cumsum(t), np.cumsum(1 - t), np.cumsum(y * t), np.cumsum(y * (1 - t))
    idx = np.unique(np.linspace(1, n, k + 1).astype(int)) - 1
    q = yt[idx] - yc[idx] * nt[idx] / np.maximum(nc[idx], 1)
    return pd.DataFrame({'share': (idx + 1) / n, 'extra_buyers_per_customer': q / n})
QINI = pd.concat([qini_points(hold.step3_tlearner.values).assign(model='First model (Step 3)'),
                  qini_points(hold.champion.values).assign(model='Champion ensemble (Step 4)'),
                  qini_points(SCORES['random']).assign(model='Random order')])
save(QINI.round(5), 'chart_qini.csv')

D = (Path('..') / 'data' / 'parquet').as_posix()
W = duckdb.sql(f"SELECT household_id, week, SUM(sales_value) spend FROM '{D}/transactions.parquet' GROUP BY 1, 2").df()
S = W.pivot_table(index='household_id', columns='week', values='spend', fill_value=0)
C = duckdb.sql(f"SELECT household_id, MAX(campaign_id = 8)::INT c8, MAX(campaign_id = 13)::INT c13, MAX(campaign_id = 18)::INT c18, MAX(campaign_id = 27)::INT c27 FROM '{D}/campaigns.parquet' GROUP BY 1").df().set_index('household_id').reindex(S.index).fillna(0)
ever = set(duckdb.sql(f"SELECT DISTINCT household_id FROM '{D}/campaigns.parquet'").df().household_id)
groups = {'Targeted by campaign 8': S.index[(C.c8 == 1) & (C.c27 == 0)], 'Never targeted': [h for h in S.index if h not in ever],
          'Targeted later': S.index[(C.c8 == 0) & (C.c27 == 0) & ((C.c13 == 1) | (C.c18 == 1))]}
WEEKS = list(range(2, 40))
CJ = pd.concat([pd.DataFrame({'week': WEEKS, 'avg_spend_usd': S.loc[ids, WEEKS].mean().values, 'group': g, 'households': len(ids)}) for g, ids in groups.items()])
save(CJ.round(2), 'chart_complete_journey_weekly.csv')

saved results\chart_qini.csv (5 KB)

saved results\chart_complete_journey_weekly.csv (4 KB)


## Part C · Cell 7 · Assumptions and sources
Every rupee figure in the app rests on these. The app shows them, labels them **Assumed**, and lets the user change them.

In [8]:
META = {
    'generated': time.strftime('%Y-%m-%d %H:%M'),
    'assumptions': {
        'basket_rs': {'value': BASKET, 'source': 'Assumption: average Indian grocery basket'},
        'gross_margin': {'value': GROSS_MARGIN, 'source': 'Indian supermarket ~15% gross margin (DMart FY21-23 annual reports)'},
        'profit_per_purchase_rs': {'value': MARGIN, 'source': 'basket x gross margin'},
        'channels_rs_per_contact': {'Email': {'value': 0.01, 'source': 'Bulk email via AWS SES, about $0.10 per 1,000'},
                                    'SMS': {'value': 0.10, 'source': 'Promotional SMS ₹0.08-0.12 (Message Central, 2026)'},
                                    'WhatsApp': {'value': 1.02, 'source': 'Meta marketing template ₹0.8631 + 18% GST (MyOperator, 2026)'},
                                    'AI voice call': {'value': 3.33, 'source': '~₹5 per minute, 40-second call (Edesy, 2026)'}},
        'cashback_redeem_share': {'value': 0.75, 'source': 'Reward breakage typically 20-30% (Stashfin)'},
        'cafe_margin': {'value': 0.75, 'source': 'Costa Coffee India FY25 gross margin 75.4%'},
        'usd_to_rs_price_level': {'value': 70, 'source': 'Price-level conversion (US tall latte ~$5, India ~₹350), not the exchange rate'},
        'per_customers': PER,
    },
    'data_notes': {'X5': 'X5 RetailHero (Russian retail), random SMS test; roubles not used: rupee values are illustrative assumptions',
                   'Starbucks': 'Simulated data released by Starbucks for analysis', 'Hillstrom': 'MineThatData e-mail test, US, public',
                   'Complete Journey': 'Dunnhumby, US, company chose whom to target (no random test)'},
    'files': sorted(str(p.relative_to(OUT)).replace('\\', '/') for p in SAVED) + ['meta.json'],
    'sources': {k: f'{v[0]} cell {v[1]}' for k, v in SOURCES.items()},
}
save(META, 'meta.json')

saved results\meta.json (5 KB)


## Cell 8 · Final checks
1. Headline numbers from CLAUDE.md are reproduced by the exported files.
2. Nothing in `app/results/` looks like customer rows: no id columns, and no table longer than the 101-point curves x 3 methods.

In [9]:
pl = lambda s: float(re.findall(r'[-+]?[\d,]+\.?\d*', s)[0].replace(',', ''))
CHECKS = {
    'Message lift +3.3 pts (2.9 to 3.8; Step 7 saved 0.0332, 0.029, 0.0375)': (round(LIFT, 4), round(SUMMARY['lift_low95'], 3), round(SUMMARY['lift_high95'], 4)) == (0.0332, 0.029, 0.0375),
    'Largest balance gap 0.0125': round(SUMMARY['largest_balance_gap'], 4) == 0.0125,
    '₹20k WhatsApp: ₹2.44 lakh (uplift) vs ₹3,300 (likely-buyer)': REPRO['₹20k WhatsApp, uplift ₹'][0] == 243600 and REPRO['₹20k WhatsApp, likely ₹'][0] == 3300,
    'SMS best plan: 95%, ₹4.96 lakh (3.3 to 6.2 lakh)': REPRO['SMS best share'][0] == '95%' and round(REPRO['SMS best profit ₹'][0], -3) == 496000
                                                         and round(REPRO['SMS range low ₹'][0], -4) == 330000 and round(REPRO['SMS range high ₹'][0], -4) == 620000,
    'Naive report overstates 8.6x (15,650 vs 1,820)': round(naive / fair, 1) == 8.6 and round(naive, -1) == 15650 and round(fair, -1) == 1820,
    'Decision flip: bottom-half AI call fair -₹0.47': round(flip, 2) == -0.47,
    'Top 10% respond 3.5x the average': round(SUMMARY['top10_vs_average'], 1) == 3.5,
    'Lift ladder 0.95 / 2.88 / 5.65': [round(100 * (v['rate_contacted'] - v['rate_held_back']), 2) for v in SUMMARY['thirds'].values()] == [0.95, 2.88, 5.65],
    'Contrast: +10.7% and -6.5%': sorted(TABLES['step10_careful']['effect %'].tolist()) == [-6.5, 10.7],
    'Retraining targeted-only -71%, recorded chance 31% -> 61%': TABLES['step9_exploration']['% of clean-test quality'].iloc[1] == -71
        and TABLES['step9_recorded_chance']['% of clean-test quality'].iloc[:2].tolist() == [31, 61],
}
bad = []
for p in OUT.rglob('*.csv'):
    t = pd.read_csv(p)
    if any('id' == c.lower() or c.lower().endswith('_id') for c in t.columns) or len(t) > 3 * 101 + 10: bad.append(p.name)
CHECKS['No customer rows or id columns in app/results'] = not bad
for k, v in CHECKS.items(): print(f'{"PASS" if v else "FAIL"}  {k}')
total = sum(p.stat().st_size for p in OUT.rglob('*') if p.is_file()) / 1024
print(f'\n{len(list(OUT.rglob("*.*")))} files, {total:,.0f} KB in app/results' + (f' | check: {bad}' if bad else ''))
print('EXPORT VERDICT:', 'PASS' if all(CHECKS.values()) else 'REVIEW')

PASS  Message lift +3.3 pts (2.9 to 3.8; Step 7 saved 0.0332, 0.029, 0.0375)
PASS  Largest balance gap 0.0125
PASS  ₹20k WhatsApp: ₹2.44 lakh (uplift) vs ₹3,300 (likely-buyer)
PASS  SMS best plan: 95%, ₹4.96 lakh (3.3 to 6.2 lakh)
PASS  Naive report overstates 8.6x (15,650 vs 1,820)
PASS  Decision flip: bottom-half AI call fair -₹0.47
PASS  Top 10% respond 3.5x the average
PASS  Lift ladder 0.95 / 2.88 / 5.65
PASS  Contrast: +10.7% and -6.5%
PASS  Retraining targeted-only -71%, recorded chance 31% -> 61%
PASS  No customer rows or id columns in app/results

44 files, 756 KB in app/results
EXPORT VERDICT: PASS
